# Content-Based Movie Recommendation System

This is a small project where I built a recommendation system that suggests similar movies based on their features (genre, director, keywords, description) - not based on user ratings or watch history. That's what makes it "content-based".

Basic idea:
1. Load a small movie dataset (`movies.csv` - I put this together myself, ~30 movies)
2. Combine each movie's features into one text field
3. Turn that text into numbers using `CountVectorizer`
4. Use cosine similarity to find which movies are "closest" to each other
5. Write a simple function that takes a movie title and returns similar movies
6. Look at the results and see if they actually make sense


## Step 1: Imports

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity


## Step 2: Load the data

In [ ]:
df = pd.read_csv("movies.csv")
print(df.shape)
df.head()


## Step 3: Feature preprocessing

Cosine similarity needs numbers, not raw text, so I need to turn each movie's features into something a model can compare. My plan:

- Combine `genre`, `director`, and `keywords` into a single text column (I'm leaving out `description` for the vectorizer since it's more free-form English text and adds noise - genre/keywords are cleaner signals of "what this movie is like").
- Lowercase everything and replace spaces in director names with nothing, so "Wren Castillo" becomes one token (`wrencastillo`) instead of two separate words that could accidentally match other people's names.
- Fill any missing values with an empty string so nothing breaks.


In [ ]:
def clean_features(df):
    df = df.copy()
    df["director_clean"] = df["director"].fillna("").str.replace(" ", "", regex=False)
    df["keywords"] = df["keywords"].fillna("")
    df["genre"] = df["genre"].fillna("")

    # combining everything into one column the vectorizer can work with
    df["combined_features"] = (
        df["genre"] + " " + df["director_clean"] + " " + df["keywords"]
    ).str.lower()

    return df


df = clean_features(df)
df[["title", "combined_features"]].head()


## Step 4: Turning text into vectors

Using `CountVectorizer` here - it just counts how many times each word (genre, director, keyword) shows up for each movie. Simple bag-of-words, but it works well for this kind of short, tag-like text.


In [ ]:
vectorizer = CountVectorizer()
feature_matrix = vectorizer.fit_transform(df["combined_features"])

print("Shape of feature matrix (movies x unique words):", feature_matrix.shape)


## Step 5: Cosine similarity

This gives me a movie x movie matrix where each cell is a similarity score between 0 (nothing in common) and 1 (identical features).


In [ ]:
similarity_matrix = cosine_similarity(feature_matrix)
print(similarity_matrix.shape)


## Step 6: The actual recommend function

In [ ]:
def recommend(title, top_n=5):
    if title not in df["title"].values:
        print(f"'{title}' not found in the dataset.")
        return

    idx = df[df["title"] == title].index[0]
    scores = list(enumerate(similarity_matrix[idx]))

    # sort by similarity score, drop the movie itself (it'll always be similarity = 1 with itself)
    scores = sorted(scores, key=lambda x: x[1], reverse=True)
    scores = [s for s in scores if s[0] != idx][:top_n]

    print(f"Because you watched: {title} ({df.loc[idx, 'genre']})\n")
    for i, score in scores:
        print(f"  {df.loc[i, 'title']:<25} | {df.loc[i, 'genre']:<8} | similarity = {score:.2f}")


In [ ]:
recommend("Iron Tide")


In [ ]:
recommend("Paper Moths")


In [ ]:
recommend("Ashfall Station")


## Step 7: Evaluating the recommendations

Since there's no user ratings to measure "accuracy" against, I'm checking this in a couple of simpler ways:
1. Do the recommended movies actually share the same genre as the movie I searched for? (A rough proxy for "does this make sense")
2. What does the overall distribution of similarity scores look like - are most movies fairly different from each other, or is everything clumped together?


In [ ]:
def genre_match_rate(title, top_n=5):
    idx = df[df["title"] == title].index[0]
    target_genre = df.loc[idx, "genre"]

    scores = list(enumerate(similarity_matrix[idx]))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)
    top_indices = [s[0] for s in scores if s[0] != idx][:top_n]

    matches = sum(df.loc[i, "genre"] == target_genre for i in top_indices)
    return matches / top_n


sample_titles = ["Iron Tide", "Paper Moths", "Ashfall Station", "Hollow Pines", "Punchline City"]

for t in sample_titles:
    rate = genre_match_rate(t)
    print(f"{t:<20} -> {rate * 100:.0f}% of top recommendations share the same genre")


In [ ]:
import matplotlib.pyplot as plt

# flatten the similarity matrix and drop the diagonal (movie compared to itself = always 1.0)
import numpy as np
scores_flat = similarity_matrix[~np.eye(similarity_matrix.shape[0], dtype=bool)]

plt.figure(figsize=(7, 5))
plt.hist(scores_flat, bins=20, color="#4c72b0", edgecolor="white")
plt.title("Distribution of Pairwise Similarity Scores")
plt.xlabel("Cosine Similarity")
plt.ylabel("Number of Movie Pairs")
plt.show()


Looking at the genre match rates, most recommendations do land in the same genre as the movie I searched for, which is a good sign that the features are actually capturing something meaningful and it's not just random.

The similarity histogram is skewed toward the lower end, meaning most movie pairs don't have much in common - which makes sense with only 30 movies across 8 genres. A handful of pairs score higher because they share both genre and director or overlapping keywords.


## Conclusion

**What I built:**
- A content-based recommender that suggests movies using their own features (genre, director, keywords) rather than needing any user rating history.
- Used `CountVectorizer` + cosine similarity to turn text features into a similarity score between every pair of movies.

**Limitations:**
- The dataset is tiny (30 movies I made up myself) - a real system would need thousands of items to give genuinely useful recommendations.
- I didn't use the `description` column at all - there's probably useful signal in there that a `TfidfVectorizer` on the full text could pick up.
- Genre match rate is a rough proxy for "good recommendation" - it doesn't account for someone wanting variety, not just more of the same genre.

**What I'd try next:**
- Add `TfidfVectorizer` on the descriptions and combine it with the current tag-based features.
- Test with a bigger, real dataset (like the MovieLens dataset) to see how this holds up at scale.
- Add a simple popularity or recency weighting so the system doesn't only recommend based on similarity.
